# Document Question Answering System (RAG)
**Project | Retrieval-Augmented Generation**

---

### Overview
This project implements a **Retrieval-Augmented Generation (RAG)** system that answers questions
based on custom documents. Instead of relying only on a language model's internal knowledge, the system:
1. **Retrieves** relevant information from documents
2. **Augments** the model input with that retrieved context
3. **Generates** answers grounded in actual data

> *"This improves factual accuracy and allows question answering over private or domain-specific data."*
> — 

### Objectives
- Understand the concept of Retrieval-Augmented Generation (RAG)
- Build a pipeline combining retrieval and generation
- Enable question answering over custom PDFs
- Learn how modern AI systems work internally

### RAG Pipeline Architecture
```
Document (PDF)
      ↓
 Stage 1. Document Ingestion   → Load & convert to raw text
      ↓
 Stage 2. Text Chunking        → Split into smaller overlapping pieces
      ↓
 Stage 3. Embedding Creation   → Convert chunks into dense vectors
      ↓
 Stage 4. Vector Database      → Store in FAISS for fast similarity search
      ↓
 Stage 5. Query Processing     → Convert user question to embedding
      ↓
 Stage 6. Context Retrieval    → Retrieve top-k most similar chunks
      ↓
 Stage 7. Answer Generation    → LLM generates grounded final answer
```

---
## Step 1: Install Dependencies

In [1]:
# Install all required libraries for the RAG pipeline
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters
!pip install -q pypdf sentence-transformers faiss-cpu
!pip install -q transformers accelerate
print("All dependencies installed successfully!")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


All dependencies installed successfully!



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Step 2: Import Libraries

In [2]:
import os
import glob
import warnings
warnings.filterwarnings("ignore")

# Document loading
from langchain_community.document_loaders import PyPDFLoader

# Text splitting (langchain 1.x: moved to langchain-text-splitters)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector store
from langchain_community.vectorstores import FAISS

# LangChain Expression Language (LCEL) — modern RAG chain
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# LLM (HuggingFace local pipeline)
from transformers import pipeline as hf_pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_huggingface import HuggingFacePipeline

print("All libraries imported successfully!")

All libraries imported successfully!


---
## Stage 1: Document Ingestion

Load the PDF document using `PyPDFLoader`.
The notebook **automatically discovers** any PDF in its working directory — no path changes needed.

> *"Documents such as PDFs or text files are loaded and converted into raw text."* — 

In [3]:
# ── Auto-discover the knowledge PDF ──────────────────────────────────────────
def find_knowledge_pdf():
    """Scan the notebook directory and return path to the knowledge PDF."""
    search_dir = os.getcwd()
    pdfs = glob.glob(os.path.join(search_dir, "*.pdf"))

    if not pdfs:
        raise FileNotFoundError(
            f"No PDF found in: {search_dir}\n"
            "Please place your knowledge PDF alongside this notebook."
        )

    # If multiple PDFs exist, skip the assignment/assignment PDF
    if len(pdfs) == 1:
        return pdfs[0]
    knowledge = [
        p for p in pdfs
        if "assignment" not in os.path.basename(p).lower()
        and "assignment" not in os.path.basename(p).lower()
    ]
    return knowledge[0] if knowledge else pdfs[0]


PDF_PATH = find_knowledge_pdf()
print(f"Knowledge PDF : {os.path.basename(PDF_PATH)}")
print(f"Full path     : {PDF_PATH}")

# ── Load with PyPDFLoader ─────────────────────────────────────────────────────
loader = PyPDFLoader(PDF_PATH)
pages  = loader.load()

print(f"\nTotal pages loaded : {len(pages)}")
print("\n" + "=" * 60)
print("Sample text from Page 1:")
print("=" * 60)
print(pages[0].page_content[:1000])
print("=" * 60)

Knowledge PDF : knowledge.pdf
Full path     : D:\test-ground\knowledge.pdf



Total pages loaded : 15

Sample text from Page 1:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispen

---
## Stage 2: Text Chunking

Split the document into smaller, overlapping chunks using `RecursiveCharacterTextSplitter`.

> *"The text is split into smaller chunks to improve retrieval accuracy."* — 

In [4]:
CHUNK_SIZE    = 500   # max characters per chunk
CHUNK_OVERLAP = 100   # overlap between consecutive chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

chunks = text_splitter.split_documents(pages)

print(f"Total chunks created : {len(chunks)}")
print(f"Chunk size           : {CHUNK_SIZE} characters")
print(f"Chunk overlap        : {CHUNK_OVERLAP} characters")
print("\n" + "=" * 60)
print("Sample Chunk #1:")
print("=" * 60)
print(chunks[0].page_content)
print(f"Metadata : {chunks[0].metadata}")
print("=" * 60)

Total chunks created : 103
Chunk size           : 500 characters
Chunk overlap        : 100 characters

Sample Chunk #1:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Metadata : {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\test-ground\\knowledge

---
## Stage 3: Embedding Creation

Convert each text chunk into a dense vector using **`sentence-transformers/all-MiniLM-L6-v2`**.
This model is free, fast, and runs entirely locally — no API key required.

> *"Each chunk is converted into a vector representation capturing its semantic meaning."* — 

In [5]:
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {EMBED_MODEL}")
print("(Downloads ~90MB on first run, cached afterwards)")

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Sanity-check embedding
test_vec = embedding_model.embed_query("What is this document about?")
print(f"\nEmbedding model ready!")
print(f"Embedding dimension    : {len(test_vec)}")
print(f"Sample values (first 5): {[round(v, 4) for v in test_vec[:5]]}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
(Downloads ~90MB on first run, cached afterwards)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16300.54it/s]


Embedding model ready!
Embedding dimension    : 384
Sample values (first 5): [-0.106, 0.168, 0.0239, 0.018, 0.0093]


---
## Stage 4: Vector Database (FAISS)

Store all embeddings in a **FAISS** (Facebook AI Similarity Search) index for sub-millisecond retrieval.

> *"Embeddings are stored in a vector database for efficient similarity search."* — 

In [6]:
print("Building FAISS vector store...")
vector_store = FAISS.from_documents(chunks, embedding_model)

print("FAISS index created!")
print(f"Total chunks stored : {vector_store.index.ntotal}")
print(f"Vector dimensions   : {vector_store.index.d}")

Building FAISS vector store...


FAISS index created!
Total chunks stored : 103
Vector dimensions   : 384


---
## Stage 5 & 6: Query Processing & Context Retrieval

> *"Stage 5: The user's question is converted into an embedding.*
> *Stage 6: The system retrieves the most relevant chunks from the database."*
> — 

In [7]:
TOP_K = 3  # retrieve top-3 most relevant chunks

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K},
)

# ── Demonstrate retrieval ─────────────────────────────────────────────────────
test_query     = "What is the main topic of this document?"
retrieved_docs = retriever.invoke(test_query)

print(f"Query    : {test_query}")
print(f"Retrieved: {len(retrieved_docs)} chunks\n")
for i, doc in enumerate(retrieved_docs, 1):
    pg = doc.metadata.get("page", 0)
    print(f"--- Chunk {i} | Page {pg + 1} ---")
    print(doc.page_content[:400])
    print()

Query    : What is the main topic of this document?
Retrieved: 3 chunks

--- Chunk 1 | Page 1 ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
G

--- Chunk 2 | Page 13 ---
Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad

--- Chunk 3 | Page 14 ---
.
<EOS>
<pad>
The
Law
will
never
be
perfect
,


---
## Stage 7a: Load Language Model (LLM)

Uses **`google/flan-t5-base`** via HuggingFace Transformers — completely free and local (~300MB).
If `GOOGLE_API_KEY` is set, switches automatically to **Gemini 1.5 Flash**.

> *"A language model generates the final answer using the retrieved context."* — 

In [8]:
LLM_MODEL = "google/flan-t5-base"

def load_llm():
    """Load LLM: Gemini (if GOOGLE_API_KEY set) or local flan-t5-base."""
    api_key = os.environ.get("GOOGLE_API_KEY", "").strip()

    if api_key:
        try:
            from langchain_google_genai import ChatGoogleGenerativeAI
            llm = ChatGoogleGenerativeAI(
                model="gemini-1.5-flash",
                google_api_key=api_key,
                temperature=0.3,
            )
            print("LLM: Google Gemini 1.5 Flash (GOOGLE_API_KEY detected)")
            return llm
        except ImportError:
            print("Note: pip install langchain-google-genai to use Gemini.")

    # ── Local fallback: flan-t5-base ──────────────────────────────────────────
    print(f"Loading local LLM: {LLM_MODEL}")
    print("(Downloading model weights on first run — ~300MB, cached after)")

    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
    model     = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL)

    # transformers 5.x compatibility — custom Seq2Seq generator wrapper
    import torch
    from langchain_core.language_models.llms import LLM
    from typing import Optional, List, Any

    class FlanT5LLM(LLM):
        """Direct flan-t5 wrapper compatible with langchain-core and transformers 5.x"""
        tokenizer: Any = None
        model: Any = None
        max_new_tokens: int = 256

        @property
        def _llm_type(self) -> str:
            return "flan-t5"

        def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs) -> str:
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                )
            return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    llm = FlanT5LLM(tokenizer=tokenizer, model=model)
    print(f"LLM ready: {LLM_MODEL} (custom wrapper)")
    return llm


llm = load_llm()

Loading local LLM: google/flan-t5-base
(Downloading model weights on first run — ~300MB, cached after)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 10407.79it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LLM ready: google/flan-t5-base (custom wrapper)


---
## Stage 7b: Build the RAG Chain (LCEL)

Modern LangChain uses **LCEL (LangChain Expression Language)** to compose pipelines.
Our chain: `Retriever → Format Context → Prompt → LLM → Parse Output`

**Example Flow:**
> User Question → Retrieves relevant sections → Provides as context → Generates concise answer

In [9]:
# ── Prompt Template ───────────────────────────────────────────────────────────
RAG_PROMPT = (
    "You are a helpful and precise assistant. "
    "Use ONLY the context below to answer the question.\n"
    "If the answer is not in the context, say: "
    "'I don't have enough information in this document.'\n\n"
    "Context:\n"
    "------------------------------------------------------------\n"
    "{context}\n"
    "------------------------------------------------------------\n\n"
    "Question: {question}\n\n"
    "Answer:"
)

prompt = PromptTemplate.from_template(RAG_PROMPT)


def format_docs(docs):
    """Concatenate retrieved document chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)


def retrieve_with_docs(query):
    """Return both retrieved docs and the formatted context string."""
    docs = retriever.invoke(query)
    return {"docs": docs, "context": format_docs(docs)}


# ── LCEL RAG Chain ────────────────────────────────────────────────────────────
rag_chain = (
    RunnableLambda(lambda q: {
        "context": format_docs(retriever.invoke(q)),
        "question": q,
    })
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled (LCEL style)!")
print("Pipeline: Question -> Retriever -> Context -> Prompt -> LLM -> Answer")

RAG chain assembled (LCEL style)!
Pipeline: Question -> Retriever -> Context -> Prompt -> LLM -> Answer


---
## Interactive Q&A: `ask_question(question)`

The function:
1. Retrieves relevant context chunks from FAISS
2. Prints the retrieved chunks with page numbers
3. Generates and prints the final grounded answer

In [10]:
def ask_question(question: str, verbose: bool = True) -> dict:
    """
    Ask a natural language question against the loaded PDF knowledge base.

    Parameters
    ----------
    question : str   - The question to answer
    verbose  : bool  - If True, prints retrieved chunks and generated answer

    Returns
    -------
    dict: {question, answer, source_documents}
    """
    if not question.strip():
        raise ValueError("Question cannot be empty.")

    # 1. Retrieve relevant chunks
    source_docs = retriever.invoke(question)

    # 2. Generate answer using RAG chain
    answer = rag_chain.invoke(question)

    if verbose:
        sep = "=" * 65
        print(f"\n{sep}")
        print(f" QUESTION : {question}")
        print(f"{sep}")

        # Print retrieved chunks
        print(f"\n RETRIEVED CONTEXT ({len(source_docs)} chunks):")
        for i, doc in enumerate(source_docs, 1):
            pg   = doc.metadata.get("page", 0)
            text = doc.page_content[:400].replace("\n", "\n    ")
            print(f"  +-- Chunk {i} | Page {pg + 1} " + "-" * 38)
            print(f"    {text}")
            print()

        # Print generated answer
        print(" GENERATED ANSWER:")
        print("-" * 65)
        print(f"  {answer}")
        print("-" * 65 + "\n")

    return {
        "question"        : question,
        "answer"          : answer,
        "source_documents": source_docs,
    }


print("ask_question() is ready!")

ask_question() is ready!


---
## Demonstration: 10 Example Questions

Ten diverse questions showcasing the full RAG pipeline.

In [11]:
demo_questions = [
    "What is this document about?",
    "Summarize the main content of the document.",
    "What are the key topics or concepts discussed?",
    "List the most important points mentioned.",
    "What conclusions or recommendations are mentioned?",
    "Who is the intended audience for this document?",
    "What problems or challenges are described?",
    "What methods or approaches are proposed?",
    "Are there any statistics, numbers, or data points mentioned?",
    "What are the future directions or next steps suggested?",
]
print(f"Running {len(demo_questions)} demonstration questions...\n")

Running 10 demonstration questions...



In [12]:
_ = ask_question(demo_questions[0])   # Q1: What is this document about?


 QUESTION : What is this document about?

 RETRIEVED CONTEXT (3 chunks):
  +-- Chunk 1 | Page 1 --------------------------------------
    Provided proper attribution is provided, Google hereby grants permission to
    reproduce the tables and figures in this paper solely for use in journalistic or
    scholarly works.
    Attention Is All You Need
    Ashish Vaswani∗
    Google Brain
    avaswani@google.com
    Noam Shazeer∗
    Google Brain
    noam@google.com
    Niki Parmar∗
    Google Research
    nikip@google.com
    Jakob Uszkoreit∗
    Google Research
    usz@google.com
    Llion Jones∗
    G

  +-- Chunk 2 | Page 13 --------------------------------------
    Attention Visualizations
    Input-Input Layer5
    It
    is
    in
    this
    spirit
    that
    a
    majority
    of
    American
    governments
    have
    passed
    new
    laws
    since
    2009
    making
    the
    registration
    or
    voting
    process
    more
    difficult
    .
    <EOS>
    <pad>

In [13]:
_ = ask_question(demo_questions[1])   # Q2: Summarize


 QUESTION : Summarize the main content of the document.

 RETRIEVED CONTEXT (3 chunks):
  +-- Chunk 1 | Page 12 --------------------------------------
    [27] Ankur Parikh, Oscar Täckström, Dipanjan Das, and Jakob Uszkoreit. A decomposable attention
    model. In Empirical Methods in Natural Language Processing , 2016.
    [28] Romain Paulus, Caiming Xiong, and Richard Socher. A deep reinforced model for abstractive
    summarization. arXiv preprint arXiv:1705.04304, 2017.
    [29] Slav Petrov, Leon Barrett, Romain Thibaux, and Dan Klein. Learning accurate, compac

  +-- Chunk 2 | Page 2 --------------------------------------
    described in section 3.2.
    Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
    of a single sequence in order to compute a representation of the sequence. Self-attention has been
    used successfully in a variety of tasks including reading comprehension, abstractive summarization,
    textual entailm

In [14]:
_ = ask_question(demo_questions[2])   # Q3: Key topics


 QUESTION : What are the key topics or concepts discussed?

 RETRIEVED CONTEXT (3 chunks):
  +-- Chunk 1 | Page 1 --------------------------------------
    Provided proper attribution is provided, Google hereby grants permission to
    reproduce the tables and figures in this paper solely for use in journalistic or
    scholarly works.
    Attention Is All You Need
    Ashish Vaswani∗
    Google Brain
    avaswani@google.com
    Noam Shazeer∗
    Google Brain
    noam@google.com
    Niki Parmar∗
    Google Research
    nikip@google.com
    Jakob Uszkoreit∗
    Google Research
    usz@google.com
    Llion Jones∗
    G

  +-- Chunk 2 | Page 12 --------------------------------------
    [27] Ankur Parikh, Oscar Täckström, Dipanjan Das, and Jakob Uszkoreit. A decomposable attention
    model. In Empirical Methods in Natural Language Processing , 2016.
    [28] Romain Paulus, Caiming Xiong, and Richard Socher. A deep reinforced model for abstractive
    summarization. arXiv preprint arXiv

In [15]:
_ = ask_question(demo_questions[3])   # Q4: Important points


 QUESTION : List the most important points mentioned.

 RETRIEVED CONTEXT (3 chunks):
  +-- Chunk 1 | Page 1 --------------------------------------
    Provided proper attribution is provided, Google hereby grants permission to
    reproduce the tables and figures in this paper solely for use in journalistic or
    scholarly works.
    Attention Is All You Need
    Ashish Vaswani∗
    Google Brain
    avaswani@google.com
    Noam Shazeer∗
    Google Brain
    noam@google.com
    Niki Parmar∗
    Google Research
    nikip@google.com
    Jakob Uszkoreit∗
    Google Research
    usz@google.com
    Llion Jones∗
    G

  +-- Chunk 2 | Page 9 --------------------------------------
    keeping the amount of computation constant, as described in Section 3.2.2. While single-head
    attention is 0.9 BLEU worse than the best setting, quality also drops off with too many heads.
    In Table 3 rows (B), we observe that reducing the attention key size dk hurts model quality. This
    suggests that

In [16]:
_ = ask_question(demo_questions[4])   # Q5: Conclusions


 QUESTION : What conclusions or recommendations are mentioned?

 RETRIEVED CONTEXT (3 chunks):
  +-- Chunk 1 | Page 1 --------------------------------------
    Provided proper attribution is provided, Google hereby grants permission to
    reproduce the tables and figures in this paper solely for use in journalistic or
    scholarly works.
    Attention Is All You Need
    Ashish Vaswani∗
    Google Brain
    avaswani@google.com
    Noam Shazeer∗
    Google Brain
    noam@google.com
    Niki Parmar∗
    Google Research
    nikip@google.com
    Jakob Uszkoreit∗
    Google Research
    usz@google.com
    Llion Jones∗
    G

  +-- Chunk 2 | Page 11 --------------------------------------
    [22] Zhouhan Lin, Minwei Feng, Cicero Nogueira dos Santos, Mo Yu, Bing Xiang, Bowen
    Zhou, and Yoshua Bengio. A structured self-attentive sentence embedding. arXiv preprint
    arXiv:1703.03130, 2017.
    [23] Minh-Thang Luong, Quoc V . Le, Ilya Sutskever, Oriol Vinyals, and Lukasz Kaiser. Multi-t

In [17]:
_ = ask_question(demo_questions[5])   # Q6: Intended audience


 QUESTION : Who is the intended audience for this document?

 RETRIEVED CONTEXT (3 chunks):
  +-- Chunk 1 | Page 1 --------------------------------------
    Provided proper attribution is provided, Google hereby grants permission to
    reproduce the tables and figures in this paper solely for use in journalistic or
    scholarly works.
    Attention Is All You Need
    Ashish Vaswani∗
    Google Brain
    avaswani@google.com
    Noam Shazeer∗
    Google Brain
    noam@google.com
    Niki Parmar∗
    Google Research
    nikip@google.com
    Jakob Uszkoreit∗
    Google Research
    usz@google.com
    Llion Jones∗
    G

  +-- Chunk 2 | Page 12 --------------------------------------
    [25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annotated
    corpus of english: The penn treebank. Computational linguistics, 19(2):313–330, 1993.
    [26] David McClosky, Eugene Charniak, and Mark Johnson. Effective self-training for parsing. In
    Proceedings

In [18]:
_ = ask_question(demo_questions[6])   # Q7: Problems/challenges


 QUESTION : What problems or challenges are described?

 RETRIEVED CONTEXT (3 chunks):
  +-- Chunk 1 | Page 13 --------------------------------------
    Attention Visualizations
    Input-Input Layer5
    It
    is
    in
    this
    spirit
    that
    a
    majority
    of
    American
    governments
    have
    passed
    new
    laws
    since
    2009
    making
    the
    registration
    or
    voting
    process
    more
    difficult
    .
    <EOS>
    <pad>
    <pad>
    <pad>
    <pad>
    <pad>
    <pad>
    It
    is
    in
    this
    spirit
    that
    a
    majority
    of
    American
    governments
    have
    passed
    new
    laws
    since
    2009
    making
    the
    registration
    or
    voting
    process
    more
    difficult
    .
    <EOS>
    <pad

  +-- Chunk 2 | Page 13 --------------------------------------
    the
    registration
    or
    voting
    process
    more
    difficult
    .
    <EOS>
    <pad>
    <pad>
    <pad>
    <pad

In [19]:
_ = ask_question(demo_questions[7])   # Q8: Methods/approaches


 QUESTION : What methods or approaches are proposed?

 RETRIEVED CONTEXT (3 chunks):
  +-- Chunk 1 | Page 11 --------------------------------------
    across languages. In Proceedings of the 2009 Conference on Empirical Methods in Natural
    Language Processing, pages 832–841. ACL, August 2009.
    [15] Rafal Jozefowicz, Oriol Vinyals, Mike Schuster, Noam Shazeer, and Yonghui Wu. Exploring
    the limits of language modeling. arXiv preprint arXiv:1602.02410, 2016.
    [16] Łukasz Kaiser and Samy Bengio. Can active memory replace attention? In Advances in Neura

  +-- Chunk 2 | Page 13 --------------------------------------
    Attention Visualizations
    Input-Input Layer5
    It
    is
    in
    this
    spirit
    that
    a
    majority
    of
    American
    governments
    have
    passed
    new
    laws
    since
    2009
    making
    the
    registration
    or
    voting
    process
    more
    difficult
    .
    <EOS>
    <pad>
    <pad>
    <pad>
    <pad>
    <pad

In [20]:
_ = ask_question(demo_questions[8])   # Q9: Statistics/data


 QUESTION : Are there any statistics, numbers, or data points mentioned?

 RETRIEVED CONTEXT (3 chunks):
  +-- Chunk 1 | Page 1 --------------------------------------
    Provided proper attribution is provided, Google hereby grants permission to
    reproduce the tables and figures in this paper solely for use in journalistic or
    scholarly works.
    Attention Is All You Need
    Ashish Vaswani∗
    Google Brain
    avaswani@google.com
    Noam Shazeer∗
    Google Brain
    noam@google.com
    Niki Parmar∗
    Google Research
    nikip@google.com
    Jakob Uszkoreit∗
    Google Research
    usz@google.com
    Llion Jones∗
    G

  +-- Chunk 2 | Page 11 --------------------------------------
    across languages. In Proceedings of the 2009 Conference on Empirical Methods in Natural
    Language Processing, pages 832–841. ACL, August 2009.
    [15] Rafal Jozefowicz, Oriol Vinyals, Mike Schuster, Noam Shazeer, and Yonghui Wu. Exploring
    the limits of language modeling. arXiv prepr

In [21]:
_ = ask_question(demo_questions[9])   # Q10: Future directions


 QUESTION : What are the future directions or next steps suggested?

 RETRIEVED CONTEXT (3 chunks):
  +-- Chunk 1 | Page 7 --------------------------------------
    We trained our models on one machine with 8 NVIDIA P100 GPUs. For our base models using
    the hyperparameters described throughout the paper, each training step took about 0.4 seconds. We
    trained the base models for a total of 100,000 steps or 12 hours. For our big models,(described on the
    bottom line of table 3), step time was 1.0 seconds. The big models were trained for 300,000 steps
    (3.5 days).
    5

  +-- Chunk 2 | Page 2 --------------------------------------
    architectures [38, 24, 15].
    Recurrent models typically factor computation along the symbol positions of the input and output
    sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
    states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
    

---
## Evaluation: Retrieved Context vs Generated Answer

Structured evaluation showing context quality and answer quality for 5 queries.

In [22]:
eval_queries = [
    "What is the main purpose of this document?",
    "Describe the methodology or approach used.",
    "What are the key findings or results?",
    "What limitations are discussed?",
    "What future improvements are suggested?",
]

eval_results = []
print("EVALUATION REPORT")
print("=" * 70)

for idx, query in enumerate(eval_queries, 1):
    r = ask_question(query, verbose=False)
    ctx = " | ".join(
        d.page_content[:100].replace("\n", " ") for d in r["source_documents"]
    )
    eval_results.append({
        "#": idx,
        "Question": query,
        "Chunks": len(r["source_documents"]),
        "Context Preview": ctx[:180] + "...",
        "Answer": r["answer"][:200],
    })
    print(f"\n[{idx}] {query}")
    print(f"    Chunks  : {len(r['source_documents'])} retrieved")
    print(f"    Context : {ctx[:120]}...")
    print(f"    Answer  : {r['answer'][:180]}")
    print("-" * 70)

print(f"\nEvaluated {len(eval_results)} queries successfully.")

EVALUATION REPORT



[1] What is the main purpose of this document?
    Chunks  : 3 retrieved
    Context : described in section 3.2. Self-attention, sometimes called intra-attention is an attention mechanism | Provided proper a...
    Answer  : to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.
----------------------------------------------------------------------



[2] Describe the methodology or approach used.
    Chunks  : 3 retrieved
    Context : described in section 3.2. Self-attention, sometimes called intra-attention is an attention mechanism | Provided proper a...
    Answer  : Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. S
----------------------------------------------------------------------



[3] What are the key findings or results?
    Chunks  : 3 retrieved
    Context : Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and | keeping the amoun...
    Answer  : We observe that reducing the attention key size dk hurts model quality. This suggests that determining compatibility is not easy and that a more sophisticated compatibility functio
----------------------------------------------------------------------



[4] What limitations are discussed?
    Chunks  : 3 retrieved
    Context : consider three desiderata. One is the total computational complexity per layer. Another is the amoun | Attention Visuali...
    Answer  : The third is the path length between long-range dependencies in the network. Learning long-range dependencies is a key challenge in many sequence transduction tasks.
----------------------------------------------------------------------



[5] What future improvements are suggested?
    Chunks  : 3 retrieved
    Context : keeping the amount of computation constant, as described in Section 3.2.2. While single-head attenti | listed in the bot...
    Answer  : making generation less sequential
----------------------------------------------------------------------

Evaluated 5 queries successfully.


In [23]:
# ── Summary Table ─────────────────────────────────────────────────────────────
print("\n" + "#" * 70)
print("  EVALUATION SUMMARY TABLE")
print("#" * 70)
print(f"{'#':<4} {'Question':<42} {'Chunks':<8} Ans.Length")
print("-" * 70)
for r in eval_results:
    q = r["Question"][:39] + "..." if len(r["Question"]) > 39 else r["Question"]
    print(f"{r['#']:<4} {q:<42} {r['Chunks']:<8} {len(r['Answer'])} chars")


######################################################################
  EVALUATION SUMMARY TABLE
######################################################################
#    Question                                   Chunks   Ans.Length
----------------------------------------------------------------------
1    What is the main purpose of this docume... 3        100 chars
2    Describe the methodology or approach us... 3        200 chars
3    What are the key findings or results?      3        200 chars
4    What limitations are discussed?            3        165 chars
5    What future improvements are suggested?    3        33 chars


---
## Key Learnings

| # | Learning |
|---|----------|
| 1 | How RAG systems combine retrieval and generation |
| 2 | Importance of retrieval in improving answer accuracy |
| 3 | Working with embeddings and vector databases |
| 4 | Handling unstructured text data |
| 5 | Designing scalable AI pipelines |

---
## Conclusion

### What is RAG?
**Retrieval-Augmented Generation (RAG)** is a hybrid AI architecture that retrieves relevant text
from a knowledge base before generating an answer — reducing hallucinations and enabling
Q&A over private, domain-specific data.

> *"RAG systems are widely used in chatbots, knowledge assistants, enterprise search systems,
> and AI-powered documentation tools."* — 

---

### Improvements & Experiments

| Improvement | Benefit |
|-------------|---------|
| **Better Chunking Strategies** | Semantic or proposition-based splitting |
| **Different Embedding Models** | `BAAI/bge-large-en-v1.5` for higher quality |
| **Hybrid Search** | BM25 keyword search + vector search combined |
| **Re-ranking** | Cross-encoder reranker for better relevance ordering |
| **Better Language Models** | Llama 3, Mistral, or Gemini Pro for richer answers |

---

### Components Used in This Project

| Stage | Component | Library | Role |
|-------|-----------|---------|------|
| 1 | `PyPDFLoader` | `langchain-community` | Document Ingestion |
| 2 | `RecursiveCharacterTextSplitter` | `langchain-text-splitters` | Text Chunking |
| 3 | `all-MiniLM-L6-v2` | `sentence-transformers` | Embedding Creation |
| 4 | `FAISS` | `faiss-cpu` | Vector Database |
| 5-6 | `FAISS.as_retriever()` | `langchain-community` | Query & Retrieval |
| 7 | `google/flan-t5-base` | `transformers` | Answer Generation |
| — | LCEL Chain | `langchain-core` | Pipeline Orchestration |

The system is **fully local, free to run, and submission-ready**.